# Visualization

In [ ]:
# lib import
import os
from datasets import get_dataset_config_names, Dataset

# setup
img_dir = os.path.join(os.getcwd(), "img")
os.makedirs(img_dir, exist_ok=True)
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "wikipos")
configs = get_dataset_config_names("whatphiliptrains/wikipos")

# subset configuration
subset_size = None


def get_subset(ds: Dataset, size: int | None = None, seed: int = 97) -> Dataset:
    ds_shuffled = ds.shuffle(seed=seed)
    if size is None:
        subset = ds_shuffled
    else:
        subset = ds_shuffled.select(range(size))
    return subset

In [ ]:
from datasets import load_dataset
from datetime import datetime
import matplotlib.pyplot as plt
import datashader as ds
from datashader.mpl_ext import dsshow, alpha_colormap
import os

for config in configs:
    print(f"Processing config: {config}")
    print("Loading data...")
    dataset = load_dataset(
        "whatphiliptrains/wikipos",
        name=config,
        cache_dir=source_ds_cache_dir,
        split="train",
    )
    subset = get_subset(dataset, subset_size)

    print("Converting subset to DataFrame...")
    df = subset.to_pandas()

    print("Generating and saving plot...")
    fig, ax = plt.subplots(figsize=(8, 8), facecolor="none")
    ax.set_facecolor("none")

    artist = dsshow(
        df,
        ds.Point("x", "y"),
        ds.count(),
        norm="log",
        cmap=alpha_colormap("#FF6600"),  # Bright orange with alpha
        x_range=(-1.0, 1.0),
        y_range=(-1.0, 1.0),
        ax=ax,
    )

    # Set tick marks and labels
    ax.set_xlim(-1.0, 1.0)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
    ax.set_yticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])

    # timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    filename = f"{config}-{subset_size}.png"
    filepath = os.path.join(img_dir, filename)
    plt.savefig(filepath, format="png", bbox_inches="tight", dpi=300, transparent=True)
    print(f"Saved plot to: {filepath}")

    plt.show()